In [8]:
import requests
import pandas as pd
from pathlib import Path
import time

In [2]:
url = "https://api.mfapi.in/mf/125497"

response = requests.get(url, timeout=30)
response.raise_for_status()

In [3]:
data = response.json()
print("API response received successfully!")
print(data["meta"]["scheme_name"])
print("Status:", data["status"])

API response received successfully!
SBI SMALL CAP FUND - Direct Plan - Growth
Status: SUCCESS


In [4]:
nav_df = pd.DataFrame(data["data"])
nav_df.head()

,date,nav
0,27-10-2026,198.74510
1,26-10-2026,195.29960
2,23-10-2026,194.12300
3,22-10-2026,197.68960
4,21-10-2026,199.13040


In [5]:
raw_folder = Path("../data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

output_path = raw_folder / "hdfc_top_100_nav.csv"

In [6]:
nav_df.to_csv(output_path, index=False)
print("Saved to:", output_path)
print("Shape:", nav_df.shape)

Saved to: ..\data\raw\hdfc_top_100_nav.csv
Shape: (3193, 2)


In [7]:
saved_df = pd.read_csv(output_path)
print(saved_df.shape)
print(saved_df.dtypes)
display(saved_df.head())

(3193, 2)
date        str
nav     float64
dtype: object


,date,nav
0,27-10-2026,198.7451
1,26-10-2026,195.2996
2,23-10-2026,194.1230
3,22-10-2026,197.6896
4,21-10-2026,199.1304


In [9]:
schemes = {
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_Large_Cap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

In [10]:
nav_data = {}

for scheme_name, scheme_code in schemes.items():
    url = f"https://api.mfapi.in/mf/{scheme_code}"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()
    nav_df = pd.DataFrame(data["data"])

    nav_data[scheme_name] = nav_df

    print(f"\n{scheme_name}")
    print("API scheme name:", data["meta"]["scheme_name"])
    print("Shape:", nav_df.shape)
    display(nav_df.head())


SBI_Bluechip
API scheme name: Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment
Shape: (3337, 2)


,date,nav
0,27-10-2026,106.54600
1,26-10-2026,106.55920
2,23-10-2026,106.49560
3,22-10-2026,106.53250
4,21-10-2026,106.68590



ICICI_Bluechip
API scheme name: Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option
Shape: (3409, 2)


,date,nav
0,27-10-2026,105.36050
1,26-10-2026,104.41530
2,23-10-2026,103.49930
3,22-10-2026,104.48360
4,21-10-2026,104.88550



Nippon_Large_Cap
API scheme name: Nippon India Large Cap Fund - Direct Plan - Growth Option
Shape: (3400, 2)


,date,nav
0,27-10-2026,94.88510
1,26-10-2026,94.59820
2,23-10-2026,94.06650
3,22-10-2026,94.84050
4,21-10-2026,95.40630



Axis_Bluechip
API scheme name: HDFC Money Market Fund - Direct Plan - Growth Option
Shape: (3666, 2)


,date,nav
0,27-10-2026,6338.96000
1,26-10-2026,6336.77700
2,23-10-2026,6332.34980
3,22-10-2026,6331.38080
4,21-10-2026,6333.44620



Kotak_Bluechip
API scheme name: Quant Mid Cap Fund - Direct Plan - Growth Option
Shape: (3403, 2)


,date,nav
0,27-10-2026,234.70450
1,26-10-2026,231.22440
2,23-10-2026,228.79330
3,22-10-2026,233.70300
4,21-10-2026,234.31950


Comparing fund master with nav history


In [19]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

In [20]:
print(fund_master.columns)
print(fund_master.shape)
fund_master.head()

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='str')
(40, 15)


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [21]:
print(fund_master.columns)

print("Fund houses:")
print(fund_master["fund_house"].dropna().unique())

print("Categories:")
print(fund_master["category"].dropna().unique())

print("Sub-categories:")
print(fund_master["sub_category"].dropna().unique())

print("Risk grades:")
print(fund_master["risk_category"].dropna().unique())

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='str')
Fund houses:
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
Categories:
<StringArray>
['Equity', 'Debt']
Length: 2, dtype: str
Sub-categories:
<StringArray>
[      'Large Cap',       'Small Cap',            'Gilt',         'Mid Cap',
  'Short Duration',           'Value',          'Liquid',       'Index/ETF',
       'Flexi Cap',           'Index', 'Large & Mid Cap',            'ELSS']
Length: 12, dtype: str
Risk grades:
<StringArray>


In [23]:
fund_codes = set(fund_master["amfi_code"].dropna().astype(str))
nav_codes = set(nav_history["amfi_code"].dropna().astype(str))

missing_codes = fund_codes - nav_codes

print("Missing scheme codes:", len(missing_codes))
print("Missing codes:", missing_codes)

Missing scheme codes: 0
Missing codes: set()
